# SQL Analysis – Olist E-Commerce
Business questions answered with SQL (SQLite in memory) on the Olist dataset. This notebook runs on the raw data; cleaning is in `notebooks/data_cleaning.ipynb`.

In [46]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Path to dataset files: /kaggle/input/brazilian-ecommerce


In [47]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")

def run(query):
    """Run a SQL query and show the result as a table."""
    return pd.read_sql_query(query, conn)

def execute(statement):
    """Run a SQL statement that doesn't return rows (CREATE, INSERT, UPDATE, DROP...)."""
    conn.executescript(statement)
    conn.commit()
    print("OK ✔")

print("SQLite version:", sqlite3.sqlite_version)

SQLite version: 3.45.1


In [48]:
customers            = pd.read_csv(f"{path}/olist_customers_dataset.csv")
orders               = pd.read_csv(f"{path}/olist_orders_dataset.csv")
order_items          = pd.read_csv(f"{path}/olist_order_items_dataset.csv")
order_payments       = pd.read_csv(f"{path}/olist_order_payments_dataset.csv")
order_reviews        = pd.read_csv(f"{path}/olist_order_reviews_dataset.csv")
products             = pd.read_csv(f"{path}/olist_products_dataset.csv")
sellers              = pd.read_csv(f"{path}/olist_sellers_dataset.csv")
category_translation = pd.read_csv(f"{path}/product_category_name_translation.csv", encoding="utf-8-sig")

In [49]:
customers.to_sql("customers", conn, index=False, if_exists="replace")
orders.to_sql("orders", conn, index=False, if_exists="replace")
order_items.to_sql("order_items", conn, index=False, if_exists="replace")
order_payments.to_sql("order_payments", conn, index=False, if_exists="replace")
order_reviews.to_sql("order_reviews", conn, index=False, if_exists="replace")
products.to_sql("products", conn, index=False, if_exists="replace")
sellers.to_sql("sellers", conn, index=False, if_exists="replace")
category_translation.to_sql("category_translation", conn, index=False, if_exists="replace")

71

## 1. What is the total revenue generated by the platform?

In [50]:
run("""
SELECT *
FROM order_payments
""")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54


In [51]:
run("""
SELECT SUM(p.payment_value) AS "Total Revenue"
FROM orders o JOIN order_payments p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
""")

,Total Revenue
0,15422461.77


# **2. How does revenue change over time (monthly)?**

In [52]:
run("""
SELECT *
FROM orders
""")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


In [53]:
run("""
SELECT strftime('%Y-%m', o.order_purchase_timestamp) AS Month, ROUND(SUM(op.payment_value), 2) AS "Revenue"
FROM orders o JOIN order_payments op ON o.order_id = op.order_id
WHERE o.order_status = 'delivered'
GROUP BY Month
ORDER BY Month
""")

,Month,Revenue
0,2016-10,46566.71
1,2016-12,19.62
2,2017-01,127545.67
3,2017-02,271298.65
4,2017-03,414369.39
5,2017-04,390952.18
6,2017-05,567066.73
7,2017-06,490225.60
8,2017-07,566403.93
9,2017-08,646000.61


# **3. Which product categories generate the highest revenue?**

In [54]:
run("""
SELECT t.product_category_name_english AS Category, ROUND(SUM(oi.price), 2) AS Revenue

FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN orders o ON oi.order_id = o.order_id
LEFT JOIN category_translation t ON p.product_category_name = t.product_category_name

WHERE o.order_status = 'delivered'

GROUP BY Category
ORDER BY Revenue DESC
LIMIT 10
""")

,Category,Revenue
0,health_beauty,1233131.72
1,watches_gifts,1166176.98
2,bed_bath_table,1023434.76
3,sports_leisure,954852.55
4,computers_accessories,888724.61
5,furniture_decor,711927.69
6,housewares,615628.69
7,cool_stuff,610204.10
8,auto,578966.65
9,toys,471286.48


# **4. How many unique customers have purchased from the platform?**

In [55]:
run("""
SELECT COUNT(DISTINCT customer_unique_id) AS "Number of Unique Customers"
FROM customers c JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
""")

,Number of Unique Customers
0,93358


## **5. How many customers are repeat buyers?**

1st step (All customers and their number of orders)

In [56]:
run("""
SELECT c.customer_unique_id AS "Customer ID" , COUNT(o.order_id) AS "Number of Orders"
FROM customers c JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_unique_id
ORDER BY COUNT(o.order_id) DESC

""")

,Customer ID,Number of Orders
0,8d50f5eadf50201ccdcedfb9e2ac8455,15
1,3e43e6105506432c953e165fb2acf44c,9
2,ca77025e7201e3b30c44b472ff346268,7
3,6469f99c1f9dfae7733b25662e7f1782,7
4,1b6c7548a2a1f9037c1fd3ddfed95f33,7
...,...,...
93353,0004aac84e0df4da2b147fca70cf8255,1
93354,0000f6ccb0745a6a4b88665a16c9f078,1
93355,0000f46a3911fa3c0805444483337064,1
93356,0000b849f77a49e4a4ce2b2a4ca5be3f,1


2nd Step Count(all more than 1)

In [57]:
run("""
SELECT COUNT(*) AS "Repeat Buyers"
FROM (
    SELECT c.customer_unique_id
    FROM customers c JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
    HAVING COUNT(*) > 1
)
""")

,Repeat Buyers
0,2801


# **6. Who are the top 10 customers based on total spending?**

In [58]:
run("""
SELECT c.customer_unique_id AS "Customer ID", ROUND(SUM(p.payment_value), 2) AS "Total Spending"

FROM customers c JOIN orders o ON c.customer_id = o.customer_id JOIN order_payments p ON o.order_id = p.order_id

WHERE o.order_status = 'delivered'

GROUP BY c.customer_unique_id
ORDER BY SUM(p.payment_value) DESC
LIMIT 10
""")

,Customer ID,Total Spending
0,0a0a92112bd4c708ca5fde585afaa872,13664.08
1,da122df9eeddfedc1dc1f5349a1a690c,7571.63
2,763c8b1c9c68a0229c42c9fc6f662b93,7274.88
3,dc4802a71eae9be1dd28f5d788ceb526,6929.31
4,459bef486812aa25204be022145caa62,6922.21
5,ff4159b92c40ebe40454e3e6a7c35ed6,6726.66
6,4007669dec559734d6f53e029e360987,6081.54
7,eebb5dda148d3893cdaf5b5ca3040ccb,4764.34
8,48e1ac109decbb87765a3eade6854098,4681.78
9,c8460e4251689ba205045f3ea17884a1,4655.91


# **7. What is the average order value (AOV)?**

In [59]:
run("""
SELECT ROUND(SUM(p.payment_value) / COUNT(DISTINCT o.order_id), 2) AS "Average Order Value"
FROM orders o JOIN order_payments p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
""")

,Average Order Value
0,159.86


## **8. What are the most common order statuses?**

In [60]:
run("""
SELECT order_status AS "Order Status", COUNT(*) AS "Number of Orders", ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM orders), 2) AS "Percentage %"
FROM orders
GROUP BY order_status
ORDER BY COUNT(*) DESC
""")

,Order Status,Number of Orders,Percentage %
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


# **Q9: How does delivery delay affect customer review scores?**

**1st ON Time**

In [61]:
run("""
SELECT COUNT(*) AS "Number of Orders", ROUND(AVG(r.review_score), 2) AS "Avg Review Score (On Time)"
FROM orders o JOIN order_reviews r ON o.order_id = r.order_id

WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date <= o.order_estimated_delivery_date
""")

,Number of Orders,Avg Review Score (On Time)
0,88653,4.29


***2nd Late***

In [62]:
run("""
SELECT COUNT(*) AS "Number of Orders", ROUND(AVG(r.review_score), 2) AS "Avg Review Score (Delayed)"
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status = 'delivered' AND o.order_delivered_customer_date > o.order_estimated_delivery_date
""")

,Number of Orders,Avg Review Score (Delayed)
0,7700,2.57


### **10. What are the most commonly used payment methods?**

In [63]:
run("""
SELECT payment_type AS "Payment Method", COUNT(*) AS "Number of Payments"
FROM order_payments
GROUP BY payment_type
ORDER BY COUNT(*) DESC
""")

,Payment Method,Number of Payments
0,credit_card,76795
1,boleto,19784
2,voucher,5775
3,debit_card,1529
4,not_defined,3
